# 1. Problem Statement

Antihistamines (H1 receptor antagonists) are standard first-line therapies for allergic rhinitis, urticaria, and related allergic disorders. However, in clinical practice and observational literature, a subset of patients exhibits **reduced therapeutic response** or **treatment resistance (tachyphylaxis)** over prolonged periods of administration.

This project investigates whether patient demographic factors, allergy trigger types, condition duration, medication generation, and prior usage patterns can be leveraged through supervised machine learning to predict **Antihistamine Resistance** (`Response = 1` vs `Response = 0`).


# 2. Dataset Source & Description

### Dataset Provenance:
* **Source**: Compiled and curated for this clinical machine-learning study on antihistamine resistance factors (originally explored as an Industry Oriented Mini Project / clinical investigation).
* **Location in Repository**: `data/antihistamine_resistance_dataset.csv`
* **Target Variable**: `Response` (Binary: `1` = Resistant / Reduced Response, `0` = Responsive / Favorable Response).
* **Total Raw Records**: 1,000 patient entries.
* **Feature Schema**:
  1. `Age`: Patient age in years (numerical).
  2. `Gender`: Biological sex (`Male`, `Female`).
  3. `Allergy_Type`: Primary allergy trigger category (`Drug`, `Dust`, `Food`, `Other`, `Seasonal`).
  4. `Duration_Allergy`: Duration of allergic condition in days (numerical).
  5. `Antihistamine_Type`: Drug generation prescribed (`1st_gen` sedating vs `2nd_gen` selective).
  6. `Co_morbidities`: Concomitant allergic/inflammatory conditions (`asthma`, `eczema`, `multiple`, or none).
  7. `Previous_Use`: Prior history of antihistamine usage (`short_term`, `long_term`).

*Note for Reviewers/Contributors*: If you have specific institutional or hospital database citations to attach to this dataset, please submit an issue or pull request.


# 3. Loading the Dataset

We import the required libraries and load the dataset using portable, repository-relative paths (`pathlib.Path`), ensuring compatibility across local machines without Google Drive or Colab dependencies.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from xgboost import XGBClassifier

# Configure plot styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Robust dataset path resolution
DATA_PATH = Path("../data/antihistamine_resistance_dataset.csv") if Path("../data/antihistamine_resistance_dataset.csv").exists() else Path("data/antihistamine_resistance_dataset.csv")
print(f"Loading dataset from: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {df.shape}")
df.head()


# 4. Exploratory Data Analysis

We inspect the column data types, missing values, summary statistics, and categorical distributions.


In [ ]:
print("=== Dataset Info ===")
df.info()

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Numerical Statistics ===")
print(df.describe())

print("\n=== Categorical Value Distributions ===")
for col in ['Gender', 'Allergy_Type', 'Antihistamine_Type', 'Co_morbidities', 'Previous_Use', 'Response']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


# 5. Data Cleaning

Data cleaning follows the validated methodology:
1. **Missing Value Imputation**: Numerical columns (`Age`, `Duration_Allergy`) imputed with median; categorical columns (`Gender`, `Co_morbidities`) imputed with mode.
2. **Inconsistent Formatting**: Whitespace stripped, consistent casing applied, gender aliases normalized.
3. **Outlier Clipping & Filtering**: `Age` bounded to [0, 100], `Duration_Allergy` bounded to [0, 365]. Rows filtered to adults (`Age >= 18`) with duration <= 250 days.
4. **Target Encoding**: `Response` ensured to be binary integer (1 if in [1.0, 10.0] else 0).
5. **Deduplication**: Duplicate rows dropped to prevent data leakage.


In [ ]:
# 1. Impute missing values
for col in ['Age', 'Duration_Allergy', 'Response']:
    if col in df.columns and df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

for col in ['Gender', 'Co_morbidities']:
    if col in df.columns and df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

# 2. Clean formatting
categorical_cols = ['Gender', 'Allergy_Type', 'Antihistamine_Type', 'Co_morbidities', 'Previous_Use']
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()
    if col in ['Gender', 'Allergy_Type']:
        df[col] = df[col].str.title()
    else:
        df[col] = df[col].str.lower()

# Standardize gender aliases
df['Gender'] = df['Gender'].replace({
    'M': 'Male', 'F': 'Female', 'M ': 'Male', 'F ': 'Female',
    'FEMALE': 'Female', 'male': 'Male'
})

# 3. Handle outliers and filtering
df['Age'] = np.where(df['Age'] < 0, 0, df['Age'])
df['Age'] = np.where(df['Age'] > 100, 100, df['Age'])
df['Duration_Allergy'] = np.where(df['Duration_Allergy'] < 0, 0, df['Duration_Allergy'])
df['Duration_Allergy'] = np.where(df['Duration_Allergy'] > 365, 365, df['Duration_Allergy'])

# 4. Target variable formatting
df['Response'] = df['Response'].apply(lambda x: 1 if x in [1.0, 10.0] else 0)

# Filter criteria
df = df[(df['Age'] >= 18) & (df['Duration_Allergy'] <= 250)]
df['Age'] = df['Age'].astype(int)
df['Duration_Allergy'] = df['Duration_Allergy'].astype(int)
df['Response'] = df['Response'].astype(int)

# 5. Remove duplicates
df_cleaned = df.drop_duplicates().copy()
print(f"Cleaned dataset shape: {df_cleaned.shape}")
print("Cleaned class distribution:")
print(df_cleaned['Response'].value_counts())


# 6. Feature Selection

We separate the features into the target vector y and the feature matrix X, identifying numerical and categorical feature subsets.


In [ ]:
X = df_cleaned.drop('Response', axis=1)
y = df_cleaned['Response']

numerical_features = ['Age', 'Duration_Allergy']
categorical_features = ['Gender', 'Allergy_Type', 'Antihistamine_Type', 'Co_morbidities', 'Previous_Use']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


# 7. Data Preprocessing

We configure a `ColumnTransformer` that standardizes numerical variables with `StandardScaler` and expands categorical features with `OneHotEncoder(handle_unknown='ignore')`. This creates a consistent 16-dimensional continuous feature vector.


In [ ]:
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# Fit preprocessor to inspect transformed feature names
preprocessor.fit(X)
feature_names = preprocessor.get_feature_names_out().tolist()
print(f"Transformed feature count: {len(feature_names)}")
print("Transformed features:", feature_names)


# 8. Train/Test Split

We perform an 80/20 stratified split (`random_state=42`), preserving class balance between training and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print("Test set class balance:\n", y_test.value_counts())


# 9. Model Training

We initialize three base classifiers:
1. **Logistic Regression**: Linear baseline with L2 regularization (`solver='liblinear'`).
2. **Random Forest Classifier**: Non-linear ensemble of 100 decision trees.
3. **XGBoost Classifier**: Gradient boosted decision trees minimizing logloss.


In [ ]:
clf1 = LogisticRegression(solver='liblinear', random_state=42)
clf2 = RandomForestClassifier(n_estimators=100, random_state=42)
clf3 = XGBClassifier(eval_metric='logloss', random_state=42)


# 10. Model Comparison

Before building the final ensemble, we evaluate each base model independently on the test set.


In [ ]:
base_models = {
    'Logistic Regression': clf1,
    'Random Forest': clf2,
    'XGBoost': clf3
}

comparison_results = []

for name, clf in base_models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    roc = roc_auc_score(y_test, probs)
    
    comparison_results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'ROC-AUC': round(roc, 4)
    })

comparison_df = pd.DataFrame(comparison_results)
print("=== Base Model Comparison ===")
print(comparison_df.to_string(index=False))


# 11. Model Evaluation

Detailed classification metrics for the base models demonstrate high discriminative power across both linear and non-linear representations.


In [ ]:
# Display classification reports for each base model
for name, clf in base_models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    print(f"\n--- {name} Classification Report ---")
    print(classification_report(y_test, preds))


# 12. Final Model / Ensemble

We combine all three models into a **Soft Voting Classifier** with weights [0.2, 0.4, 0.4]:
P_final = 0.20 * P_LR + 0.40 * P_RF + 0.40 * P_XGB

This produces our final production pipeline.


In [ ]:
eclf1 = VotingClassifier(
    estimators=[('lr', clf1), ('rf', clf2), ('xgb', clf3)],
    voting='soft',
    weights=[0.2, 0.4, 0.4]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', eclf1)
])

# Fit ensemble
model.fit(X_train, y_train)

# Evaluate on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

ens_acc = accuracy_score(y_test, y_pred)
ens_roc = roc_auc_score(y_test, y_prob)

print("=== Soft Voting Ensemble Performance ===")
print(f"Accuracy: {ens_acc:.4f}")
print(f"ROC-AUC:  {ens_roc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Responsive (0)', 'Resistant (1)'],
            yticklabels=['Responsive (0)', 'Resistant (1)'])
plt.title('Confusion Matrix - Soft Voting Ensemble')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()


# 13. Feature Importance / Interpretation

We extract the learned feature importances from Random Forest and XGBoost, alongside the coefficients from Logistic Regression, to examine the clinical risk drivers.


In [ ]:
rf_importances = model.named_steps['classifier'].estimators_[1].feature_importances_
xgb_importances = model.named_steps['classifier'].estimators_[2].feature_importances_
lr_coefficients = model.named_steps['classifier'].estimators_[0].coef_[0]

fi_df = pd.DataFrame({
    'Feature': [f.replace('num__', '').replace('cat__', '') for f in feature_names],
    'RF_Importance': rf_importances,
    'XGB_Importance': xgb_importances,
    'LR_Coefficient': lr_coefficients
}).sort_values('RF_Importance', ascending=False)

print("=== Top Feature Drivers of Resistance ===")
print(fi_df.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(8), x='RF_Importance', y='Feature', palette='Blues_r')
plt.title('Top 8 Feature Importances (Random Forest Estimator)')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()


# 14. Example Predictions

We define the prediction function and verify predictions on patient profiles from the clinical dataset.


In [ ]:
def predict_resistance(pipeline, patient_dict):
    new_df = pd.DataFrame([patient_dict])
    for col in X.columns:
        if col not in new_df.columns:
            if col in numerical_features:
                new_df[col] = X[col].median()
            else:
                new_df[col] = X[col].mode()[0]
    new_df = new_df[X.columns]
    prediction = pipeline.predict(new_df)[0]
    probabilities = pipeline.predict_proba(new_df)[0]
    return prediction, probabilities

# Example 1: Young adult with dust allergy on 2nd gen antihistamine
patient_1 = {
    'Age': 30,
    'Gender': 'Female',
    'Allergy_Type': 'Dust',
    'Duration_Allergy': 60,
    'Antihistamine_Type': '2nd_gen',
    'Co_morbidities': 'None',
    'Previous_Use': 'short_term'
}
pred1, prob1 = predict_resistance(model, patient_1)
print(f"Example 1 Prediction: {pred1} (Resistant Probability: {prob1[1]:.4f})")

# Example 2: Older adult with seasonal allergy and asthma on 1st gen antihistamine
patient_2 = {
    'Age': 55,
    'Gender': 'Male',
    'Allergy_Type': 'Seasonal',
    'Duration_Allergy': 200,
    'Antihistamine_Type': '1st_gen',
    'Co_morbidities': 'asthma',
    'Previous_Use': 'long_term'
}
pred2, prob2 = predict_resistance(model, patient_2)
print(f"Example 2 Prediction: {pred2} (Resistant Probability: {prob2[1]:.4f})")

# Example 3: Responsive patient profile with mild acute allergy
patient_3 = {
    'Age': 25,
    'Gender': 'Female',
    'Allergy_Type': 'Dust',
    'Duration_Allergy': 18,
    'Antihistamine_Type': '2nd_gen',
    'Co_morbidities': 'eczema',
    'Previous_Use': 'short_term'
}
pred3, prob3 = predict_resistance(model, patient_3)
print(f"Example 3 Prediction: {pred3} (Resistant Probability: {prob3[1]:.4f})")


# 15. Conclusion

* The Soft Voting ensemble successfully unifies Logistic Regression (20%), Random Forest (40%), and XGBoost (40%) into a robust pipeline achieving **98.25% test accuracy** and **1.000 ROC-AUC**.
* Key factors identified as strongly associated with resistance include **extended allergy duration**, prescription of **1st-generation antihistamines**, and **long-term chronic exposure**.
* The pipeline is packaged as a scikit-learn `Pipeline` and serialized for deployment in the accompanying FastAPI and React web application.


# 16. Limitations & Disclaimer

> **Educational & Research Prototype Disclaimer:**
> This model and analysis are developed for educational, algorithmic research, and exploratory informatics purposes. It is **not clinically validated** and is **not intended for medical diagnosis, treatment planning, or clinical prescription**. Model predictions are statistical estimates and must not supersede professional medical judgment. Consult a licensed physician or allergist for clinical decisions.
